In [1]:
!pip install langchain chromadb openai tiktoken pypdf langchain_openai langchain-community

  Using cached backoff-2.2.1-py3-none-any.whl.metadata (14 kB)
  Using cached zipp-3.23.0-py3-none-any.whl.metadata (3.6 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
   ---------------------------------------- 0.0/21.9 MB ? eta -:--:--
   - -------------------------------------- 1.0/21.9 MB 4.8 MB/s eta 0:00:05
   --- ------------------------------------ 1.8/21.9 MB 4.2 MB/s eta 0:00:05
   ---- ----------------------------------- 2.6/21.9 MB 4.2 MB/s eta 0:00:05
   ------ --------------------------------- 3.7/21.9 MB 4.5 MB/s eta 0:00:05
   --------- ------------------------------ 5.0/21.9 MB 4.8 MB/s eta 0:00:04
   ----------- ---------------------------- 6.0/21.9 MB 4.8 MB/s eta 0:00:04
   ------------ --------------------------- 7.1/21.9 MB 4.8 MB/s eta 0:00:04
   -------------- ------------------------- 7.9/21.9 MB 4.7 M


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
!pip install langchain_chroma


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# ---------------------------------------------------------
# Import OpenAIEmbeddings
# ---------------------------------------------------------
# OpenAIEmbeddings is used to convert text into numerical vectors
# (called embeddings). These vectors represent the semantic meaning
# of text and are used for similarity search, clustering, and RAG systems.
#
# Example:
# "AI is powerful" → [0.12, -0.45, 0.98, ...]  (vector representation)
#
# These embeddings help machines understand meaning instead of
# just matching keywords.
from langchain_openai import OpenAIEmbeddings


# ---------------------------------------------------------
# Import Chroma Vector Store
# ---------------------------------------------------------
# Chroma is a vector database used to store embeddings.
#
# It allows:
# 1. Saving document embeddings
# 2. Performing similarity search
# 3. Retrieving relevant documents based on meaning
#
# Workflow:
# Text → Embeddings → Stored in Chroma → Semantic Search
#
# Commonly used in:
# - RAG (Retrieval Augmented Generation)
# - Question Answering Systems
# - Document Search Applications
import os
from langchain_chroma import Chroma

In [2]:
# ---------------------------------------------------------
# Import Document class
# ---------------------------------------------------------
# Document is a LangChain object used to store:
# 1. page_content → actual text data
# 2. metadata → additional information about the text
#
# Documents are later converted into embeddings and stored
# inside a vector database (like Chroma) for semantic search.
from langchain_core.documents import Document


# ---------------------------------------------------------
# Create LangChain Documents for IPL Players
# ---------------------------------------------------------
# Each Document represents one piece of knowledge.
# page_content contains the descriptive text.
# metadata stores structured information (here: IPL team name).
# Metadata helps in filtering search results later.

# Document 1 → Virat Kohli
doc1 = Document(
    page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
    metadata={"team": "Royal Challengers Bangalore"}   # Extra information about the document
)

# Document 2 → Rohit Sharma
doc2 = Document(
    page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
    metadata={"team": "Mumbai Indians"}
)

# Document 3 → MS Dhoni
doc3 = Document(
    page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
    metadata={"team": "Chennai Super Kings"}
)

# Document 4 → Jasprit Bumrah
doc4 = Document(
    page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
    metadata={"team": "Mumbai Indians"}
)

# Document 5 → Ravindra Jadeja
doc5 = Document(
    page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
    metadata={"team": "Chennai Super Kings"}
)

In [3]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings 
#Hugging face embedding model
embedding =  HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

In [9]:
# ---------------------------------------------------------
# Create a Chroma Vector Store
# ---------------------------------------------------------
# Chroma is a vector database used to store embeddings
# and perform semantic similarity searches.
#
# When documents are added:
#   Text → Embeddings → Stored inside Chroma DB
#
# Later, queries are also converted into embeddings and
# matched against stored vectors to find similar content.

vector_store = Chroma(

    embedding_function=embedding,
    
    # -----------------------------------------------------
    # persist_directory
    # -----------------------------------------------------
    # Location on disk where the database will be saved.
    # This allows the vector database to persist even after
    # the program stops running.
    #
    # If the folder already exists, Chroma loads existing data.
    persist_directory='my_chroma_db',

    # -----------------------------------------------------
    # collection_name
    # -----------------------------------------------------
    # Name of the collection (similar to a table in SQL).
    # Multiple collections can exist inside the same DB,
    # each storing different types of documents.
    collection_name='sample'
)

In [10]:
# add documents
vector_store.add_documents(docs)

['5145c487-6386-4951-8d13-e7f670d6742d',
 '4c65d065-b8f2-489c-b6c5-bc755261a009',
 '3f93c507-d4e7-4fd7-8e83-7f83b8078293',
 '2c00867c-2ab7-494a-ae85-e43d53e2a928',
 '0bdf54d3-5799-48c6-9c8e-9fa80479e831']

In [11]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['5145c487-6386-4951-8d13-e7f670d6742d',
  '4c65d065-b8f2-489c-b6c5-bc755261a009',
  '3f93c507-d4e7-4fd7-8e83-7f83b8078293',
  '2c00867c-2ab7-494a-ae85-e43d53e2a928',
  '0bdf54d3-5799-48c6-9c8e-9fa80479e831'],
 'embeddings': array([[ 0.00994726,  0.06914339, -0.05147114, ..., -0.03543339,
          0.0128481 ,  0.0124829 ],
        [ 0.00127745,  0.0312985 , -0.02375377, ..., -0.00518358,
         -0.03280613,  0.02737715],
        [-0.10265917,  0.02650809,  0.02271502, ..., -0.03359748,
         -0.07984944, -0.01507708],
        [ 0.02123395, -0.02468547, -0.04494376, ..., -0.1099581 ,
          0.0057256 ,  0.09915379],
        [ 0.0187398 ,  0.04382846, -0.04304254, ..., -0.07801617,
         -0.07840686, -0.0030419 ]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [12]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(id='2c00867c-2ab7-494a-ae85-e43d53e2a928', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='4c65d065-b8f2-489c-b6c5-bc755261a009', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.")]

In [14]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a best cricketer?',
    k=2
)

[(Document(id='5145c487-6386-4951-8d13-e7f670d6742d', metadata={'team': 'Royal Challengers Bangalore'}, page_content='Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.'),
  0.8594334125518799),
 (Document(id='4c65d065-b8f2-489c-b6c5-bc755261a009', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  0.8974161148071289)]

In [15]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='3f93c507-d4e7-4fd7-8e83-7f83b8078293', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8436007499694824),
 (Document(id='0bdf54d3-5799-48c6-9c8e-9fa80479e831', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.890937328338623)]

In [17]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='5145c487-6386-4951-8d13-e7f670d6742d', document=updated_doc1)

# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])


{'ids': ['5145c487-6386-4951-8d13-e7f670d6742d',
  '4c65d065-b8f2-489c-b6c5-bc755261a009',
  '3f93c507-d4e7-4fd7-8e83-7f83b8078293',
  '2c00867c-2ab7-494a-ae85-e43d53e2a928',
  '0bdf54d3-5799-48c6-9c8e-9fa80479e831'],
 'embeddings': array([[-0.00233745,  0.05902084, -0.04774046, ..., -0.07264046,
          0.00276784, -0.00344087],
        [ 0.00127745,  0.0312985 , -0.02375377, ..., -0.00518358,
         -0.03280613,  0.02737715],
        [-0.10265917,  0.02650809,  0.02271502, ..., -0.03359748,
         -0.07984944, -0.01507708],
        [ 0.02123395, -0.02468547, -0.04494376, ..., -0.1099581 ,
          0.0057256 ,  0.09915379],
        [ 0.0187398 ,  0.04382846, -0.04304254, ..., -0.07801617,
         -0.07840686, -0.0030419 ]], shape=(5, 384)),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple ce

In [18]:
# delete document
vector_store.delete(ids=['5145c487-6386-4951-8d13-e7f670d6742d'])

# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['4c65d065-b8f2-489c-b6c5-bc755261a009',
  '3f93c507-d4e7-4fd7-8e83-7f83b8078293',
  '2c00867c-2ab7-494a-ae85-e43d53e2a928',
  '0bdf54d3-5799-48c6-9c8e-9fa80479e831'],
 'embeddings': array([[ 0.00127745,  0.0312985 , -0.02375377, ..., -0.00518358,
         -0.03280613,  0.02737715],
        [-0.10265917,  0.02650809,  0.02271502, ..., -0.03359748,
         -0.07984944, -0.01507708],
        [ 0.02123395, -0.02468547, -0.04494376, ..., -0.1099581 ,
          0.0057256 ,  0.09915379],
        [ 0.0187398 ,  0.04382846, -0.04304254, ..., -0.07801617,
         -0.07840686, -0.0030419 ]], shape=(4, 384)),
 'documents': ["Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Jasprit Bumrah is

In [ ]:
s